# Pokrycie kodu testami czarnoskrzynkowymi

W tym zadaniu sprawdzamy, jaką część kodu wykonują testy czarnoskrzynkowe z poprzedniego zadania (`01_testy_czarnoskrzynkowe.ipynb`).
Nie dopisywaliśmy nowych testów.

## Poprawki w testach

Testy uruchamiały się bez zmian, więc nic w nich nie poprawialiśmy. Trzeba było tylko doinstalować `pytest-cov`:

```
pip install -e .[dev] pytest-cov
```

## Jak mierzyliśmy

Użyliśmy coverage.py przez wtyczkę pytest-cov. Mierzymy pokrycie linii (instrukcji) i gałęzi (`--cov-branch`).
Opcja `--cov-context=test` zapisuje dla każdej linii, które testy ją wykonały. Dzięki temu da się sprawdzić, ile pokrywa każdy test osobno.

Raport dla całych plików nie ma dla nas sensu, bo np. `backend/utils.py` ma ponad 700 instrukcji, a testujemy z niego kilka funkcji.
Dlatego liczymy pokrycie tylko dla funkcji, które testowaliśmy (B1-B26). Doliczamy też funkcje pomocnicze, które są przez nie wywoływane,
np. `_count_tasks` w B11 albo funkcje wewnętrzne w B17 i B18.

Komórka poniżej uruchamia testy z pomiarem pokrycia, zapisuje raport `coverage.json` i raport HTML w folderze `htmlcov/`.
W raporcie HTML (`htmlcov/index.html`) widać na kolorowo, które linie zostały wykonane. Po kliknięciu numeru linii widać, który test ją wykonał.

In [14]:
!cd ../.. && python -m pytest tests/unit/blackbox -q -p no:cacheprovider -W ignore::DeprecationWarning --color=yes \
    --cov=backend --cov=mcp_server --cov=plugins --cov-branch --cov-context=test --cov-report=
!cd ../.. && python -m coverage json -q -o ZADANIA_ZAJECIOWE/zajecia_2/coverage.json && python -m coverage html -q --show-contexts -d ZADANIA_ZAJECIOWE/zajecia_2/htmlcov

........................................................................ [ 30%]
..........................................xx............................ [ 60%]
........................................................................ [ 90%]
.......................                                                  [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\fastapi\testclient.py:1
  d:\paola\Studia\sem 7\badanie jakosci\NoteDiscovery-qa-testing\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
237 passed, 2 xfailed, 1 warning in 2.49s


## Osiągnięte pokrycie

Dla każdej funkcji: ile linii i gałęzi wykonały testy, a na końcu numery niepokrytych linii i gałęzi.
Zapis `881->882` oznacza przejście z linii 881 do 882, które nie wystąpiło w żadnym teście.

In [15]:
import json
from pathlib import Path

ROOT = Path("../..").resolve()
raport = json.load(open("coverage.json"))
# na Windowsie coverage zapisuje ścieżki z "\\", ujednolicamy do "/"
raport["files"] = {k.replace("\\", "/"): v for k, v in raport["files"].items()}

# funkcje, które testowaliśmy, razem z funkcjami pomocniczymi, które wywołują
FUNKCJE = {
    "B1":  ("backend/utils.py", ["paginate", "PaginationResult.to_dict"]),
    "B2":  ("backend/utils.py", ["sanitize_filename"]),
    "B3":  ("backend/utils.py", ["get_media_type"]),
    "B4":  ("backend/utils.py", ["path_has_hash"]),
    "B5":  ("backend/export.py", ["parse_image_size_spec"]),
    "B6":  ("backend/export.py", ["attachment_content_disposition"]),
    "B7":  ("backend/export.py", ["strip_frontmatter"]),
    "B8":  ("backend/export.py", ["generate_media_placeholder"]),
    "B9":  ("backend/main.py", ["_normalize_share_public_origin"]),
    "B10": ("backend/main.py", ["safe_error_message"]),
    "B11": ("plugins/note_stats.py", ["Plugin.calculate_stats", "Plugin._count_tasks"]),
    "B12": ("mcp_server/config.py", ["load_config", "MCPConfig.__post_init__", "MCPConfig.headers"]),
    "B13": ("mcp_server/server.py", ["MCPServer._validate_path"]),
    "B14": ("mcp_server/server.py", ["MCPServer._format_response"]),
    "B15": ("mcp_server/tools.py", ["get_tool_names", "get_tool_by_name"]),
    "B16": ("backend/utils.py", ["parse_tags"]),
    "B17": ("backend/utils.py", ["apply_template_placeholders", "apply_template_placeholders._strftime_sub"]),
    "B18": ("backend/export.py", ["convert_wikilinks_to_html", "convert_wikilinks_to_html.stash",
                                  "convert_wikilinks_to_html.replace_wikilink"]),
    "B19": ("backend/note_index.py", ["extract_links_from_content"]),
    "B20": ("backend/note_index.py", ["extract_search_terms"]),
    "B21": ("backend/themes.py", ["parse_theme_metadata"]),
    "B22": ("backend/main.py", ["verify_api_key"]),
    "B23": ("backend/main.py", ["verify_password"]),
    "B24": ("plugins/contrib/search_open_tasks.py", ["Plugin._open_tasks"]),
    "B25": ("backend/share.py", ["validate_slug"]),
    "B26": ("backend/share.py", ["generate_token"]),
}

def dane_funkcji(bid):
    plik, nazwy = FUNKCJE[bid]
    linie, brak_linii, galezie, brak_galezi = set(), set(), set(), set()
    for n in nazwy:
        f = raport["files"][plik]["functions"][n]
        linie |= set(f["executed_lines"]) | set(f["missing_lines"])
        brak_linii |= set(f["missing_lines"])
        galezie |= {tuple(b) for b in f["executed_branches"] + f["missing_branches"]}
        brak_galezi |= {tuple(b) for b in f["missing_branches"]}
    return plik, linie, brak_linii, galezie, brak_galezi

suma = [0, 0, 0, 0]
print(f"{'ID':4} {'funkcja':32} {'linie':>7} {'gałęzie':>8}   niepokryte linie / gałęzie")
for bid, (plik, nazwy) in FUNKCJE.items():
    _, linie, bl, gal, bg = dane_funkcji(bid)
    suma = [suma[0] + len(linie) - len(bl), suma[1] + len(linie), suma[2] + len(gal) - len(bg), suma[3] + len(gal)]
    brak = ", ".join(map(str, sorted(bl))) + (" | " + ", ".join(f"{a}->{b}" for a, b in sorted(bg)) if bg else "")
    print(f"{bid:4} {nazwy[0]:32} {len(linie)-len(bl):>3}/{len(linie):<3} {len(gal)-len(bg):>3}/{len(gal):<4}   {brak}")

print()
print(f"Razem linie:   {suma[0]}/{suma[1]} = {100*suma[0]/suma[1]:.1f}%")
print(f"Razem gałęzie: {suma[2]}/{suma[3]} = {100*suma[2]/suma[3]:.1f}%")

ID   funkcja                            linie  gałęzie   niepokryte linie / gałęzie
B1   paginate                          13/13    4/4      
B2   sanitize_filename                 10/11    3/4      882 | 881->882
B3   get_media_type                     7/8     5/6      997 | 996->997
B4   path_has_hash                      1/1     0/0      
B5   parse_image_size_spec             19/19    9/10      | 81->89
B6   attachment_content_disposition     4/4     2/2      
B7   strip_frontmatter                 12/13    9/10     152 | 151->152
B8   generate_media_placeholder         6/6     0/0      
B9   _normalize_share_public_origin     7/7     4/4      
B10  safe_error_message                 5/5     2/2      
B11  Plugin.calculate_stats            44/48   13/18     141, 142, 143, 144 | 140->141, 141->142, 141->146, 142->141, 142->143
B12  load_config                       20/20    6/6      
B13  MCPServer._validate_path          11/11   10/10     
B14  MCPServer._format_response         5/

**Niepokrytych zostało łącznie 13 linii (300 − 287) i 12 gałęzi (138 − 126).**
Razem testy wykonują 287 z 300 linii (95,7%) i 126 ze 138 gałęzi (91,3%) w testowanych funkcjach.
19 z 26 funkcji ma 100% pokrycia linii i gałęzi. Braki są w siedmiu: B2, B3, B5, B7, B11, B16 i B22 (w B5 brakuje tylko jednej gałęzi).

W wierszu B11 liczymy razem `Plugin.calculate_stats` i wywoływaną przez nią `Plugin._count_tasks`.
Samo `calculate_stats` ma 100% pokrycia, a wszystkie braki (linie 141-144) są w `_count_tasks`.

## Linie, które nie zostały wykonane

In [16]:
# kod linii, które nie zostały wykonane
for bid in FUNKCJE:
    plik, _, bl, _, bg = dane_funkcji(bid)
    if not bl and not bg:
        continue
    kod = (ROOT / plik).read_text(encoding="utf-8").splitlines()
    print(f"--- {bid} ({plik})")
    for nr in sorted(bl | {a for a, _ in bg}):
        znacznik = "  brak" if nr in bl else "  częściowo"
        print(f"{nr:5}{znacznik:12}{kod[nr-1].rstrip()}")

--- B2 (backend/utils.py)
  881  częściowo     if not name:
  882  brak              name = 'unnamed'
--- B3 (backend/utils.py)
  996  częściowo     if name_lower.startswith('drawing-') and name_lower.endswith('.png'):
  997  brak              return 'drawing'
--- B5 (backend/export.py)
   81  częściowo         if solo:
--- B7 (backend/export.py)
  151  częściowo     if lines[0].strip() != '---':
  152  brak              return content
--- B11 (plugins/note_stats.py)
  140  częściowo         if lines and lines[0].strip() == '---':
  141  brak                  for i in range(1, len(lines)):
  142  brak                      if lines[i].strip() == '---':
  143  brak                          start = i + 1
  144  brak                          break
--- B16 (backend/utils.py)
 1038  częściowo         if lines[0].strip() != '---':
 1039  brak                  return tags
 1078  częściowo                     if tag:
 1080  częściowo                 elif stripped and not stripped.startswith('#'

## Fragmenty nieosiągalne dla danych zgodnych ze specyfikacją

Dla każdego niepokrytego fragmentu sprawdziliśmy, czy da się do niego dojść poprawnymi danymi wejściowymi.

| ID | Linie / gałęzie | Osiągalne? | Dane, które by je wykonały / uzasadnienie |
|---|---|---|---|
| B2 | 881->882, 882 | tak | nazwa złożona z samych znaków zabronionych, np. `"***.md"` -> `"unnamed.md"` |
| B3 | 996->997, 997 | tak | `"drawing-1.png"` -> `'drawing'` (typ opisany w docstringu) |
| B5 | 81->89 | tak | `allow_solo=True` i tekst, który nie jest samym rozmiarem, np. `"caption\|100"` |
| B7 | 151->152, 152 | tak | pusta linia przed frontmatterem, np. `"\n---\na: 1\n---\ntekst"` |
| B11 | 140->141, 141-144 | tak | notatka z frontmatterem, np. `"---\ntitle: x\n---\n- [ ] zadanie"` |
| B16 | 1038->1039, 1039 | tak | jak w B7: pusta linia przed `---` |
| B16 | 1078->1055 | tak | pusty element listy tagów: `"tags:\n  -\n  - a"` |
| B16 | 1080->1055 | tak | komentarz w liście tagów: `"tags:\n  # komentarz\n  - a"` |
| **B16** | **1087-1089** (`except`) | **nie** | Linie 1087-1089 to blok `except` w funkcji `parse_tags` (odczytuje tagi z nagłówka notatki). Wykonałyby się tylko wtedy, gdyby kod w bloku `try` rzucił wyjątek. Tymczasem w `try` są wyłącznie proste operacje na tekście: `split`, `strip`, `startswith` i wycinanie fragmentów. Żadna z nich nie zgłasza błędu dla jakiegokolwiek tekstu, nawet pustego. Według specyfikacji wejściem jest zawsze tekst notatki (`str`), więc poprawnymi danymi nie da się tu wejść. |
| **B22** | **638-639** (`except`) | **nie** | Linie 638-639 to blok `except` w funkcji `verify_api_key` (sprawdza, czy klucz API podany w żądaniu zgadza się z kluczem z konfiguracji). Wykonałyby się tylko wtedy, gdyby kod w bloku `try` rzucił wyjątek. W `try` są dwa kroki: zamiana obu kluczy z tekstu na bajty (`.encode('utf-8')`) i ich porównanie (`secrets.compare_digest`). Porównanie dwóch ciągów bajtów nigdy nie zgłasza błędu. Zamiana na bajty zgłasza błąd tylko dla tekstu z niepoprawnym znakiem, tzw. samotnym surogatem, którego nie da się zapisać w UTF-8. Klucz trafia do funkcji z nagłówka HTTP, a serwer (Starlette) odczytuje nagłówki w kodowaniu latin-1, w którym każdy bajt to zwykły, poprawny znak. |

Wniosek: z 13 niepokrytych linii **5 (1087-1089 i 638-639) jest nieosiągalnych dla danych zgodnych ze specyfikacją**.
Pozostałe 8 linii i wszystkie 12 gałęzi da się pokryć poprawnymi danymi.
Maksymalne pokrycie osiągalne poprawnymi danymi to 295/300 linii (98,3%) i 138/138 gałęzi.

## Testy odpowiedzialne za poszczególne gałęzie

Dzięki `--cov-context=test` plik `.coverage` przechowuje osobno dane każdego testu (również każdego przypadku testu sparametryzowanego).
Poniżej dla każdej gałęzi testowanych funkcji wypisujemy testy, które ją wykonały.
Testy sparametryzowane zwijamy do nazwy funkcji testowej, a w nawiasie podajemy, ile ich przypadków przeszło daną gałęzią.
Funkcje bez instrukcji warunkowych (B4, B8, B18, B19, B20, B26) nie mają gałęzi, więc ich tu nie ma.

In [17]:
import re
from collections import defaultdict
import coverage

# wczytujemy surowe dane z pliku .coverage (z kontekstami testów)
cov = coverage.Coverage(data_file=str(ROOT / ".coverage"))
cov.load()
dane = cov.get_data()
TESTY = sorted(c for c in dane.measured_contexts() if c.endswith("|run"))

def nazwa(test):
    # "tests/unit/blackbox/plik.py::test_x[param]|run" -> "test_x[param]"
    return test.split("::", 1)[1].removesuffix("|run")

# dla każdego testu osobno: które linie i gałęzie każdego pliku wykonał
# (_analyze liczy gałęzie tak samo jak raport coverage, np. wyjście z pętli w "with")
WYKONANE = {}
for t in TESTY:
    dane.set_query_contexts([re.escape(t) + "$"])
    WYKONANE[t] = {}
    for plik in {p for p, _ in FUNKCJE.values()}:
        a = cov._analyze(str(ROOT / plik))
        WYKONANE[t][plik] = (set(a.executed), set(a.arcs_executed))
dane.set_query_contexts(None)

def testy_galezi(plik, galaz):
    return [t for t in TESTY if galaz in WYKONANE[t][plik][1]]

def skrot(testy):
    # testy sparametryzowane zwijamy do nazwy funkcji z liczbą przypadków
    licz = defaultdict(int)
    for t in testy:
        licz[nazwa(t).split("[")[0]] += 1
    return ", ".join(f"{n} (x{k})" if k > 1 else n for n, k in sorted(licz.items()))

print(f"Liczba testów (kontekstów): {len(TESTY)}\n")
for bid in FUNKCJE:
    plik, _, _, gal, _ = dane_funkcji(bid)
    if not gal:
        continue
    print(f"--- {bid} {FUNKCJE[bid][1][0]}")
    for g in sorted(gal):
        kto = testy_galezi(plik, g)
        print(f"  {g[0]:>5}->{g[1]:<5} {skrot(kto) if kto else 'BRAK - żaden test'}")

Liczba testów (kontekstów): 239

--- B1 paginate
    100->101   test_paginate_sort_and_to_dict
    100->104   test_paginate_bounds (x7), test_paginate_no_limit_returns_all
    104->106   test_paginate_bounds (x7), test_paginate_sort_and_to_dict
    104->112   test_paginate_no_limit_returns_all
--- B2 sanitize_filename
    861->862   test_sanitize_filename_keeps_unicode_and_edge_cases
    861->865   test_sanitize_filename_keeps_unicode_and_edge_cases, test_sanitize_filename_removes_forbidden (x11)
    881->882   BRAK - żaden test
    881->885   test_sanitize_filename_keeps_unicode_and_edge_cases, test_sanitize_filename_removes_forbidden (x11)
--- B3 get_media_type
    996->997   BRAK - żaden test
    996->998   test_get_media_type (x7)
    999->1000  test_get_media_type (x7)
    999->1002  test_get_media_type (x2)
   1000->999   test_get_media_type (x5)
   1000->1001  test_get_media_type (x5)
--- B5 parse_image_size_spec
     76->77    test_parse_image_size_spec
     76->78    test_pars

Obserwacje:
- Niektóre gałęzie wykonuje **tylko jeden test (jeden przypadek)**, np. B1 `100->101` (tylko `test_paginate_sort_and_to_dict`),
  B2 `861->862` (pusta nazwa), B6 obie gałęzie, B16 `1067->1069`, `1075->1080` i `1080->1082`, B24 `136->137` (ucięcie długiego tekstu).
  Gdyby ten jeden test usunąć lub zmienić, gałąź przestałaby być testowana.
- W B7, B10, B14 i B15 wszystkie gałęzie pokrywa jedna funkcja testowa, która sprawdza kilka przypadków po kolei.
  Pokrycie jest pełne, ale jeśli pierwsza asercja nie przejdzie, kolejne gałęzie nie zostaną już sprawdzone.
- Gałęzie oznaczone `BRAK` to te same 12 gałęzi, które wyszły w tabeli pokrycia.
- Zapis `29->-22` w B12 oznacza wyjście z funkcji (funkcja zaczyna się w linii 22).

## Testy wykonujące te same fragmenty kodu

Dla każdej funkcji grupujemy testy, które wykonują w niej **dokładnie ten sam zbiór linii i gałęzi** (tę samą ścieżkę).
Jeśli kilka testów jest w jednej grupie, to z punktu widzenia pokrycia są zamienne: usunięcie wszystkich poza jednym nie zmieni wyniku coverage.

In [18]:
# testy, które w danej funkcji wykonują dokładnie te same linie i gałęzie
lacznie = 0
for bid in FUNKCJE:
    plik, linie, _, gal, _ = dane_funkcji(bid)
    grupy = defaultdict(list)
    for t in TESTY:
        l, g = WYKONANE[t][plik]
        slad = (frozenset(l & linie), frozenset(g & gal))
        if slad[0]:                      # test w ogóle wszedł do funkcji
            grupy[slad].append(t)
    wszystkie = sum(len(v) for v in grupy.values())
    nadmiar = wszystkie - len(grupy)
    lacznie += nadmiar
    print(f"{bid:4} {FUNKCJE[bid][1][0]:32} testów: {wszystkie:3}  różnych ścieżek: {len(grupy):2}  powtórzeń: {nadmiar}")
    for slad, kto in sorted(grupy.items(), key=lambda x: -len(x[1])):
        if len(kto) > 1:
            pokaz = ", ".join(nazwa(t) for t in kto[:4]) + (f", ... (+{len(kto)-4})" if len(kto) > 4 else "")
            print(f"       {len(kto):3} x ta sama ścieżka: {pokaz}")
print(f"\nRazem powtórzeń (suma po funkcjach): {lacznie}")

B1   paginate                         testów:   9  różnych ścieżek:  3  powtórzeń: 6
         7 x ta sama ścieżka: test_paginate_bounds[0-0-expected5-True], test_paginate_bounds[10-0-expected6-False], test_paginate_bounds[3--5-expected4-True], test_paginate_bounds[3-0-expected0-True], ... (+3)
B2   sanitize_filename                testów:  12  różnych ścieżek:  2  powtórzeń: 10
        11 x ta sama ścieżka: test_sanitize_filename_removes_forbidden["], test_sanitize_filename_removes_forbidden[*], test_sanitize_filename_removes_forbidden[/], test_sanitize_filename_removes_forbidden[:], ... (+7)
B3   get_media_type                   testów:   7  różnych ścieżek:  3  powtórzeń: 4
         3 x ta sama ścieżka: test_get_media_type[a.mp3-audio], test_get_media_type[a.mp4-video], test_get_media_type[a.pdf-document]
         2 x ta sama ścieżka: test_get_media_type[a.JPG-image], test_get_media_type[a.png-image]
         2 x ta sama ścieżka: test_get_media_type[a.xyz-None], test_get_media_type[n

Obserwacje:
- W B4, B8, B19, B20 i B26 **wszystkie testy wykonują identyczny kod**, bo te funkcje nie mają rozgałęzień (wyrażenie regularne, `secrets`, f-string).
  Coverage nie odróżnia tu testów od siebie. Mimo to nie są one zbędne, bo każdy sprawdza inną klasę danych (np. różne rodzaje linków w B19).
- Największe grupy to B17 (20 przypadków `apply_template_placeholders` na tej samej ścieżce), B24 (10 i 8), B25 (8 i 7) oraz B2 (11 znaków zabronionych).
  Różnice między nimi leżą w danych przetwarzanych przez wyrażenia regularne, a nie w przepływie sterowania. Z punktu widzenia pokrycia
  są nadmiarowe, ale np. 11 znaków w B2 chroni przed błędem w klasie znaków regexa. To sprawdzimy w zadaniu z mutacjami.
- Faktycznie zbędne wydają się np. `test_config_defaults` i `test_config_strips_trailing_slash` (B12) oraz
  `test_stats_empty_note` i `test_stats_reading_time_grows_with_words` (B11): wykonują ten sam kod i różnią się tylko sprawdzaną wartością.
- Z drugiej strony w B21 (7 ścieżek na 8 testów) i B13 (6 na 8) testy są dobrze rozłożone na różne ścieżki.

## Wnioski

- Testy czarnoskrzynkowe pokrywają 95,7% linii i 91,3% gałęzi testowanych funkcji. To dużo, bo pisaliśmy je bez patrzenia w kod,
  ale wybraliśmy raczej małe funkcje z dobrym opisem.
- Dwa bloki `except` (B16, B22, razem 5 linii) są nieosiągalne dla poprawnych danych. Do 100% pokrycia linii nie da się więc dojść bez podawania danych niezgodnych ze specyfikacją. Gałęzie da się pokryć w 100%.
- Funkcje oparte na wyrażeniach regularnych (B19, B20, B25) mają 100% pokrycia już po jednym teście. Dla nich pokrycie linii prawie nic nie mówi o jakości testów.
- Pokrycie per test pokazało, że wiele gałęzi zależy od jednego przypadku testowego, a wiele testów idzie dokładnie tą samą ścieżką. Liczba testów nie przekłada się więc wprost na pokrycie.
